# NB-R01 — Clean Data Splits (Leakage-Safe Train/Val/Test Construction)

**Pipeline stage:** 1 of 13

**Purpose.** Build the feature set and chronological train/validation/test splits used by every downstream notebook in this pipeline. This is the foundation notebook: every later result (model training, statistical tests, trading simulation) depends on the splits produced here being free of label leakage.

**Why this matters.** The prediction target is a 21-trading-day-ahead direction label. A naive chronological split can leak information across split boundaries: an observation near the end of the training window can have its label computed from a price that falls inside the validation window, and likewise between validation and test. This notebook removes that leakage by trimming the last 21 trading days before each split boundary before computing labels, so no label in any split depends on a price observed in a later split.

**Inputs:** `data/raw/market_data.csv` (Bank Nifty OHLCV), `data/raw/india_vix.csv` (India VIX).

**Outputs:** `data/processed/train.csv`, `val.csv`, `test.csv`, `scaler.joblib`, `feature_cols.json`, `split_summary.csv`.

**Method summary:**
- 16 leakage-safe technical indicators computed causally (each value at date *t* uses only data up to and including *t*): MACD (12-26-9), EMA-20, RSI-14 (Wilder), Stochastic %K/%D, ROC-10, Bollinger Bands (20, 2σ), ATR-14 (Wilder), and 1/2/3/5-day log-return lags.
- Forward direction labels at 1, 5, and 21 trading days.
- Chronological split with the last 21 rows of each pre-boundary period trimmed to prevent label leakage across the split.
- Feature scaling (`StandardScaler`) fit on the training split only, then applied to validation and test without refitting.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJ = Path('..').resolve()  # repo root, assuming this notebook is run from notebooks/
RAW  = PROJ / 'data' / 'raw'
PROC = PROJ / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

HORIZON = 21   # trading days forward
print(f'Horizon: {HORIZON} trading days')

Horizon: 21 trading days


## 1. Load and Merge Raw Data

In [2]:
# Load Bank Nifty OHLCV
bn = pd.read_csv(RAW / 'market_data.csv', parse_dates=['date'])
bn.columns = [c.strip().lower().replace(' ', '_') for c in bn.columns]
print('Bank Nifty columns:', bn.columns.tolist())
print('Bank Nifty shape:', bn.shape)
print(bn.head(3))

Bank Nifty columns: ['date', 'open', 'high', 'low', 'close', 'volume']
Bank Nifty shape: (2526, 6)
        date          open          high           low         close  volume
0 2016-01-01  16932.303970  17067.251627  16823.856007  17039.052734   29200
1 2016-01-04  16966.151973  16966.151973  16575.256135  16598.957031   52300
2 2016-01-05  16651.858107  16670.006335  16474.658986  16542.308594   53600


In [3]:
# Load India VIX
vix = pd.read_csv(RAW / 'india_vix.csv', parse_dates=['date'])
vix.columns = [c.strip().lower().replace(' ', '_') for c in vix.columns]
print('VIX columns:', vix.columns.tolist())
print('VIX shape:', vix.shape)
print(vix.head(3))

VIX columns: ['date', 'vix_close', 'vix_high', 'vix_low']
VIX shape: (2510, 4)
        date  vix_close  vix_high  vix_low
0 2016-01-01  14.260000     14.35    13.11
1 2016-01-04  16.840000     17.10    14.03
2 2016-01-05  16.700001     16.84    15.57


In [4]:
# Columns are already lowercase from loading step
# VIX close column is 'vix_close'
vix_close_col = [c for c in vix.columns if 'close' in c or ('vix' in c.lower() and c != 'date')][0]
print(f'VIX close column identified: {vix_close_col}')

vix_slim = vix[['date', vix_close_col]].rename(columns={vix_close_col: 'india_vix'})

# Merge on date
df = bn.merge(vix_slim, on='date', how='inner')
df = df.sort_values('date').reset_index(drop=True)
print(f'Merged shape: {df.shape}')
print(f'Date range: {df.date.min()} -> {df.date.max()}')
print(df[['date','india_vix']].head(3))

VIX close column identified: vix_close
Merged shape: (2510, 7)
Date range: 2016-01-01 00:00:00 -> 2026-03-30 00:00:00
        date  india_vix
0 2016-01-01  14.260000
1 2016-01-04  16.840000
2 2016-01-05  16.700001


## 2. Feature Engineering — 16 Leakage-Safe Technical Indicators

In [5]:
# Identify OHLCV columns
close_col = [c for c in df.columns if 'close' in c and 'india' not in c][0]
high_col  = [c for c in df.columns if 'high' in c][0]
low_col   = [c for c in df.columns if 'low' in c][0]
vol_col   = [c for c in df.columns if 'vol' in c or 'volume' in c]
vol_col   = vol_col[0] if vol_col else None

print(f'Close: {close_col}, High: {high_col}, Low: {low_col}, Volume: {vol_col}')

close = df[close_col]
high  = df[high_col]
low   = df[low_col]

Close: close, High: high, Low: low, Volume: volume


In [6]:
# ------------------------------------------------------------------
# Feature 1-3: MACD (12-26-9 standard)
# EMA uses exponential weighting; all computations up to and including t only
# ------------------------------------------------------------------
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
df['macd']         = ema12 - ema26
df['macd_signal']  = df['macd'].ewm(span=9, adjust=False).mean()
df['macd_hist']    = df['macd'] - df['macd_signal']

# Feature 4: EMA-20
df['ema20'] = close.ewm(span=20, adjust=False).mean()

# Feature 5: RSI-14
delta = close.diff()
gain  = delta.clip(lower=0)
loss  = (-delta).clip(lower=0)
avg_gain = gain.ewm(com=13, adjust=False).mean()   # Wilder smoothing: com = period-1
avg_loss = loss.ewm(com=13, adjust=False).mean()
rs = avg_gain / avg_loss.replace(0, np.nan)
df['rsi14'] = 100 - (100 / (1 + rs))

# Features 6-7: Stochastic Oscillator %K and %D (14-day lookback, 3-day smoothing)
lowest14  = low.rolling(14).min()
highest14 = high.rolling(14).max()
df['stoch_k'] = 100 * (close - lowest14) / (highest14 - lowest14).replace(0, np.nan)
df['stoch_d'] = df['stoch_k'].rolling(3).mean()

# Feature 8: ROC-10 (Rate of Change)
df['roc10'] = close.pct_change(10) * 100

# Features 9-11: Bollinger Bands (20-day SMA, 2 std)
bb_mid   = close.rolling(20).mean()
bb_std   = close.rolling(20).std()
df['bb_upper'] = bb_mid + 2 * bb_std
df['bb_lower'] = bb_mid - 2 * bb_std
df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / bb_mid

# Feature 12: ATR-14
tr = pd.concat([
    high - low,
    (high - close.shift(1)).abs(),
    (low  - close.shift(1)).abs()
], axis=1).max(axis=1)
df['atr14'] = tr.ewm(com=13, adjust=False).mean()  # Wilder smoothing

# Features 13-16: Log-return lags
# Convention: log_ret_lag_k = log(P[t] / P[t-k]) -- backward looking, no leakage
for k in [1, 2, 3, 5]:
    df[f'log_ret_lag{k}'] = np.log(close / close.shift(k))

FEATURE_COLS = [
    'macd', 'macd_signal', 'macd_hist', 'ema20',
    'rsi14', 'stoch_k', 'stoch_d', 'roc10',
    'bb_upper', 'bb_lower', 'bb_width', 'atr14',
    'log_ret_lag1', 'log_ret_lag2', 'log_ret_lag3', 'log_ret_lag5'
]
print(f'Total features: {len(FEATURE_COLS)}')
print('Features:', FEATURE_COLS)

Total features: 16
Features: ['macd', 'macd_signal', 'macd_hist', 'ema20', 'rsi14', 'stoch_k', 'stoch_d', 'roc10', 'bb_upper', 'bb_lower', 'bb_width', 'atr14', 'log_ret_lag1', 'log_ret_lag2', 'log_ret_lag3', 'log_ret_lag5']


## 3. Create Forward Labels
Label for row `t` = 1 if `close[t+21] > close[t]` else 0.
Rows where `t + 21` is beyond the dataset end will have NaN labels.

In [7]:
df['close_fwd21'] = close.shift(-HORIZON)
df['dir_21d']     = (df['close_fwd21'] > close).astype(float)
df.loc[df['close_fwd21'].isna(), 'dir_21d'] = np.nan

# Also create 1d and 5d labels for completeness
for h in [1, 5]:
    fwd = close.shift(-h)
    df[f'dir_{h}d'] = (fwd > close).astype(float)
    df.loc[fwd.isna(), f'dir_{h}d'] = np.nan

print('Label distribution (dir_21d):'); print(df['dir_21d'].value_counts())
print(f'NaN labels (end of series): {df["dir_21d"].isna().sum()}')

Label distribution (dir_21d):
dir_21d
1.0    1554
0.0     935
Name: count, dtype: int64
NaN labels (end of series): 21


## 4. Chronological Splits — Trimming Boundary Rows to Prevent Label Leakage

The 21-day-ahead label for a row at date *t* is only well-defined once `close[t+21]` is known. For rows within the last 21 trading days of the training window, `close[t+21]` falls inside the validation window; the same issue occurs between validation and test. To keep every split's labels fully self-contained, the last `HORIZON=21` rows of the training and validation splits are dropped before saving.


In [8]:
# Original boundary dates (inclusive)
TRAIN_START = '2016-05-23'
TRAIN_END   = '2023-10-12'
VAL_START   = '2023-10-20'
VAL_END     = '2025-01-06'
TEST_START  = '2025-01-14'
TEST_END    = '2026-03-30'

# Get all dates as a sorted array
all_dates = df['date'].sort_values().values

def effective_end(split_end_str, horizon, all_dates):
    """Return the date that is `horizon` trading days BEFORE split_end.
    The rows from (effective_end+1) to split_end are dropped from this split.
    """
    split_end = pd.Timestamp(split_end_str)
    dates_up_to_end = all_dates[all_dates <= split_end]
    if len(dates_up_to_end) <= horizon:
        return None
    effective = pd.Timestamp(dates_up_to_end[-(horizon + 1)])
    dropped_n = horizon
    return effective, dropped_n

eff_train_end, n_drop_train = effective_end(TRAIN_END, HORIZON, all_dates)
eff_val_end,   n_drop_val   = effective_end(VAL_END,   HORIZON, all_dates)

print(f'Train: {TRAIN_START} -> {eff_train_end.date()}  (dropped last {n_drop_train} rows)')
print(f'Val:   {VAL_START}   -> {eff_val_end.date()}  (dropped last {n_drop_val} rows)')
print(f'Test:  {TEST_START}  -> {TEST_END}  (no trailing truncation; NaN-label rows dropped naturally)')

Train: 2016-05-23 -> 2023-09-11  (dropped last 21 rows)
Val:   2023-10-20   -> 2024-12-04  (dropped last 21 rows)
Test:  2025-01-14  -> 2026-03-30  (no trailing truncation; NaN-label rows dropped naturally)


In [9]:
# Drop rows with NaN features (warm-up period for rolling indicators)
df_clean = df.dropna(subset=FEATURE_COLS + ['dir_21d', 'india_vix']).copy()
print(f'After dropping NaN rows: {df_clean.shape}')

# Build splits
train = df_clean[
    (df_clean['date'] >= TRAIN_START) &
    (df_clean['date'] <= eff_train_end)
].copy()

val = df_clean[
    (df_clean['date'] >= VAL_START) &
    (df_clean['date'] <= eff_val_end)
].copy()

test = df_clean[
    (df_clean['date'] >= TEST_START) &
    (df_clean['date'] <= TEST_END)
].copy()

print(f'Train rows: {len(train)} | {train.date.min().date()} -> {train.date.max().date()}')
print(f'Val rows:   {len(val)}   | {val.date.min().date()} -> {val.date.max().date()}')
print(f'Test rows:  {len(test)}  | {test.date.min().date()} -> {test.date.max().date()}')

# Verify no date overlap
assert len(set(train.date) & set(val.date)) == 0, 'Train/Val date overlap!'
assert len(set(val.date) & set(test.date)) == 0, 'Val/Test date overlap!'
print('\n[OK] No date overlap between splits.')

After dropping NaN rows: (2470, 27)
Train rows: 1794 | 2016-05-23 -> 2023-09-11
Val rows:   273   | 2023-10-20 -> 2024-12-04
Test rows:  276  | 2025-01-14 -> 2026-02-25

[OK] No date overlap between splits.


In [10]:
# Class balance per split
for name, split in [('Train', train), ('Val', val), ('Test', test)]:
    up_pct = split['dir_21d'].mean() * 100
    print(f'{name}: {len(split)} rows | Up%={up_pct:.1f}%')

Train: 1794 rows | Up%=62.0%
Val: 273 rows | Up%=64.8%
Test: 276 rows | Up%=68.8%


## 5. Feature Scaling
StandardScaler fitted on **training set only**. Applied to val and test without refitting.

In [11]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
train[FEATURE_COLS] = scaler.fit_transform(train[FEATURE_COLS])
val[FEATURE_COLS]   = scaler.transform(val[FEATURE_COLS])
test[FEATURE_COLS]  = scaler.transform(test[FEATURE_COLS])

# Save scaler
scaler_path = PROC / 'scaler.joblib'
joblib.dump(scaler, scaler_path)
print(f'Scaler saved: {scaler_path}')

Scaler saved: F:\MLSAPU\PhD-SPPU\India-VIX-Major-Revision\data\processed\scaler.joblib


## 6. Save Clean Splits

In [12]:
COLS_TO_SAVE = ['date', close_col, 'india_vix', 'dir_1d', 'dir_5d', 'dir_21d'] + FEATURE_COLS

train[COLS_TO_SAVE].to_csv(PROC / 'train.csv', index=False)
val[COLS_TO_SAVE].to_csv(PROC / 'val.csv', index=False)
test[COLS_TO_SAVE].to_csv(PROC / 'test.csv', index=False)

# Save feature list
import json
with open(PROC / 'feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)

print('Saved:')
for fname in ['train.csv', 'val.csv', 'test.csv', 'feature_cols.json']:
    p = PROC / fname
    print(f'  {p.name}: {p.stat().st_size:,} bytes')

Saved:
  train.csv: 666,260 bytes
  val.csv: 100,453 bytes
  test.csv: 101,720 bytes
  feature_cols.json: 245 bytes


## 7. Summary Table

In [13]:
summary = []
for name, split, orig_end in [
    ('Train', train, TRAIN_END),
    ('Val',   val,   VAL_END),
    ('Test',  test,  TEST_END)
]:
    summary.append({
        'Split':       name,
        'Start':       split.date.min().strftime('%Y-%m-%d'),
        'End (clean)': split.date.max().strftime('%Y-%m-%d'),
        'Rows':        len(split),
        'Up% (21d)':   f"{split['dir_21d'].mean()*100:.1f}%",
        'Note':        f'Last {HORIZON} days of boundary removed' if name != 'Test' else 'NaN-label rows excluded'
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(PROC / 'split_summary.csv', index=False)
print('\nSplit summary saved.')

Split      Start End (clean)  Rows Up% (21d)                             Note
Train 2016-05-23  2023-09-11  1794     62.0% Last 21 days of boundary removed
  Val 2023-10-20  2024-12-04   273     64.8% Last 21 days of boundary removed
 Test 2025-01-14  2026-02-25   276     68.8%          NaN-label rows excluded

Split summary saved.


---
## Summary

**Pipeline stage:** 1 of 13 (see `notebooks/README.md` for the full pipeline map).

**Artifacts produced by this notebook:**

- `data/processed/train.csv`
- `data/processed/val.csv`
- `data/processed/test.csv`
- `data/processed/scaler.joblib`
- `data/processed/feature_cols.json`
- `data/processed/split_summary.csv`

**Next notebook:** `NB-R02_vix_threshold_recalibration.ipynb`
